# 04 — Business Insights & Retention Strategy
**Bank Retention Intelligence Platform**

This notebook covers:
- Customer segmentation (KMeans, 4 clusters)
- SHAP explainability analysis
- Retention Strength Index (RSI) computation
- Rule-based recommendation engine
- High-value disengaged customer analysis
- Revenue at risk quantification
- Final retention strategies

> **Input:** `data/processed/features_dataset.csv` + `models/best_model.pkl`  
> **Output:** `data/processed/final_segmented_dataset.csv`, `data/processed/segment_insights.csv`

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.clustering import segment_customers, compute_rsi, get_recommendation, SEG_FEATURES
from src.visualization import (plot_clusters_pca, plot_segment_profiles,
                                plot_rsi_distribution)
from src.utils import load_processed, save_processed, load_model, save_figure

print("Libraries loaded")

Libraries loaded


## 1. Load Data & Model

In [2]:
df = load_processed('features_dataset.csv')
model, scaler, feature_cols = load_model('best_model.pkl')

print(f"Dataset : {df.shape[0]:,} rows")
print(f"Model   : {type(model).__name__}")
print(f"Features: {len(feature_cols)}")

Dataset : 10,000 rows
Model   : CatBoostClassifier
Features: 15


## 2. KMeans Customer Segmentation

In [3]:
df, seg_summary, X_sc, km = segment_customers(df, n_clusters=4)

print("Segmentation complete. Cluster breakdown:")
print("="*65)
print(seg_summary.to_string(index=False))
print("="*65)

Segmentation complete. Cluster breakdown:
           Cluster  Count  ChurnRate  AvgBalance  AvgAge  AvgProducts  ActiveRate
         High Risk   2003       20.7     11412.0    37.2         1.82         0.0
     Premium Loyal   3009       16.0    121886.0    39.7         1.32       100.0
Wealthy Disengaged   2846       31.2    123186.0    38.6         1.32         0.0
      Young Active   2142       11.9     10969.0    39.8         1.84       100.0


  File "e:\bank-retention-intelligence\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\sanch\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sanch\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\sanch\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


## 3. Segment Visualisation — PCA

In [4]:
fig = plot_clusters_pca(X_sc, df['Cluster'].tolist())
save_figure(fig, 'customer_segments_pca.png')
plt.show()

  Figure saved → e:\bank-retention-intelligence\reports/screenshots\customer_segments_pca.png


## 4. Segment Profiles Chart

In [5]:
fig = plot_segment_profiles(seg_summary)
save_figure(fig, 'segment_profiles.png')
plt.show()

  Figure saved → e:\bank-retention-intelligence\reports/screenshots\segment_profiles.png


## 5. Segment Deep Dive

In [6]:
COLORS = {'Young Active':'#3266ad','Premium Loyal':'#1d9e75',
          'Wealthy Disengaged':'#f0a500','High Risk':'#c0392b'}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
for i, (_, row) in enumerate(seg_summary.iterrows()):
    color = COLORS.get(row['Cluster'], '#73726c')
    metrics = ['ChurnRate','AvgBalance','AvgProducts','ActiveRate']
    values  = [row[m] for m in metrics]
    labels  = ['Churn %','Avg Balance €','Avg Products','Active %']
    bars = axes[i].bar(labels, values, color=color, edgecolor='white', width=0.55, alpha=0.85)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                     f'{val:,.0f}', ha='center', fontsize=9, fontweight='bold')
    axes[i].set_title(f"{row['Cluster']}  (n={int(row['Count']):,})",
                      fontsize=12, fontweight='bold', color=color)
    axes[i].tick_params(axis='x', labelsize=9)

fig.suptitle('Segment Profiles — Key Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'segment_deep_dive.png')
plt.show()

  Figure saved → e:\bank-retention-intelligence\reports/screenshots\segment_deep_dive.png


## 6. SHAP Explainability

In [7]:
try:
    import shap

    X_enc = df[feature_cols].fillna(0)
    X_sc_all = scaler.transform(X_enc)
    idx = np.random.choice(len(X_sc_all), 1000, replace=False)
    X_sample = X_sc_all[idx]

    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_sample)
        if isinstance(sv, list):
            sv = sv[1]
    except Exception:
        bg = shap.sample(X_sc_all, 100, random_state=42)
        explainer = shap.KernelExplainer(model.predict_proba, bg)
        sv = explainer.shap_values(X_sample)
        if isinstance(sv, list):
            sv = sv[1]

    X_sample_df = pd.DataFrame(X_sample, columns=feature_cols)
    plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, X_sample_df, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary — Global Churn Drivers', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/screenshots/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    shap_df = pd.DataFrame({
        'Feature': feature_cols,
        'MeanAbsSHAP': np.abs(sv).mean(axis=0)
    }).sort_values('MeanAbsSHAP', ascending=False)

    print("Top 8 global churn drivers (SHAP):")
    print(shap_df.head(8).to_string(index=False))

except ImportError:
    print("shap not installed. Install: pip install shap")
    print("Showing feature importance as proxy:")
    if hasattr(model, 'feature_importances_'):
        imp = pd.DataFrame({'Feature': feature_cols,
                            'Importance': model.feature_importances_
                           }).sort_values('Importance', ascending=False)
        print(imp.head(8).to_string(index=False))

Top 8 global churn drivers (SHAP):
        Feature  MeanAbsSHAP
            Age     1.037832
  NumOfProducts     0.880545
         Tenure     0.578657
  Geography_enc     0.409702
     Gender_enc     0.283366
 IsActiveMember     0.231665
        Balance     0.222704
EngagementScore     0.167943


## 7. Compute Churn Probabilities for All Customers

In [8]:
X_all = df[feature_cols].fillna(0)
X_all_sc = scaler.transform(X_all)
df['ChurnProbability'] = model.predict_proba(X_all_sc)[:, 1].round(4)
df['PredictedChurn']   = (df['ChurnProbability'] > 0.5).astype(int)

print(f"Churn probabilities computed for all {len(df):,} customers")
print(f"Predicted churners (prob > 0.5): {df['PredictedChurn'].sum():,}")
print(f"\nProbability distribution:")
print(df['ChurnProbability'].describe().round(4))

Churn probabilities computed for all 10,000 customers
Predicted churners (prob > 0.5): 1,717

Probability distribution:
count    10000.0000
mean         0.2398
std          0.2730
min          0.0008
25%          0.0436
50%          0.1170
75%          0.3389
max          0.9994
Name: ChurnProbability, dtype: float64


## 8. Retention Strength Index (RSI)

In [9]:
df = compute_rsi(df)

rsi_summary = df.groupby('RSICategory').agg(
    Count    =('RSI','count'),
    AvgRSI   =('RSI','mean'),
    ChurnRate=('Exited', lambda x: round(x.mean()*100,1))
).reset_index()

cat_order = ['High Risk','Moderate Risk','Stable','Loyal']
rsi_summary = rsi_summary.set_index('RSICategory').reindex(
    [c for c in cat_order if c in rsi_summary['RSICategory'].values]
).reset_index()

print("RSI Category Breakdown:")
print("="*50)
print(rsi_summary.round(1).to_string(index=False))
print("="*50)

fig = plot_rsi_distribution(df)
save_figure(fig, 'rsi_distribution.png')
plt.show()

RSI Category Breakdown:
  RSICategory  Count  AvgRSI  ChurnRate
    High Risk   1290    22.0       40.0
Moderate Risk   4621    47.0       24.2
       Stable   2793    69.9       11.9
        Loyal   1296    89.5        5.4
  Figure saved → e:\bank-retention-intelligence\reports/screenshots\rsi_distribution.png


## 9. Recommendation Engine

In [10]:
df['Recommendation'] = df.apply(get_recommendation, axis=1)

rec_summary = df.groupby('Recommendation').agg(
    Count    =('Exited','count'),
    ChurnRate=('Exited', lambda x: round(x.mean()*100,1)),
    AvgProb  =('ChurnProbability', lambda x: round(x.mean()*100,1)),
    AvgBal   =('Balance', lambda x: round(x.mean(),0))
).reset_index().sort_values('ChurnRate', ascending=False)

print("Recommendation Engine Output:")
print("="*70)
print(rec_summary.to_string(index=False))
print("="*70)

fig, ax = plt.subplots(figsize=(9,4))
colors_rec = {'Immediate Outreach':'#c0392b','Reactivation Campaign':'#f0a500',
              'Cross-Sell Programme':'#3266ad',
              'Relationship Manager Assignment':'#7f77dd',
              'Standard Retention Programme':'#1d9e75'}
bars = ax.barh(rec_summary['Recommendation'], rec_summary['Count'],
               color=[colors_rec.get(r,'#73726c') for r in rec_summary['Recommendation']],
               edgecolor='white', height=0.55)
for bar, val in zip(bars, rec_summary['Count']):
    ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Number of Customers')
ax.set_title('Customers by Recommended Action', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'recommendation_distribution.png')
plt.show()

Recommendation Engine Output:
                 Recommendation  Count  ChurnRate  AvgProb   AvgBal
             Immediate Outreach    793       95.3     91.6  92974.0
           Cross-Sell Programme   2503       17.1     22.3  98676.0
          Reactivation Campaign   4220       16.7     21.5  74839.0
Relationship Manager Assignment    823       10.1     13.4 131403.0
   Standard Retention Programme   1661        3.9      5.7  11450.0
  Figure saved → e:\bank-retention-intelligence\reports/screenshots\recommendation_distribution.png


## 10. High-Value Disengaged Customer Deep Dive

In [11]:
median_bal = df['Balance'].median()
hv = df[(df['Balance'] > median_bal) & (df['IsActiveMember'] == 0)]
hv_churned = hv[hv['Exited'] == 1]

print("HIGH-VALUE DISENGAGED ANALYSIS")
print("="*50)
print(f"  Total HV disengaged    : {len(hv):,}")
print(f"  Churn rate             : {hv['Exited'].mean()*100:.1f}%")
print(f"  Overall churn rate     : {df['Exited'].mean()*100:.1f}%")
print(f"  Risk premium           : +{(hv['Exited'].mean()-df['Exited'].mean())*100:.1f} pp")
print(f"  Avg balance            : €{hv['Balance'].mean():,.0f}")
print(f"  Total revenue @ risk   : €{hv_churned['Balance'].sum()/1e6:.1f}M")
print(f"  Germany share          : {(hv['Geography']=='Germany').mean()*100:.1f}%")
print("="*50)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
comp = pd.DataFrame({'Segment':['Overall','HV Disengaged'],
                     'ChurnRate':[df['Exited'].mean()*100, hv['Exited'].mean()*100]})
axes[0].bar(comp['Segment'], comp['ChurnRate'],
            color=['#3266ad','#c0392b'], width=0.45, edgecolor='white')
for i,(_, row) in enumerate(comp.iterrows()):
    axes[0].text(i, row['ChurnRate']+0.4, f"{row['ChurnRate']:.1f}%",
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)'); axes[0].set_ylim(0,40)
axes[0].set_title('HV Disengaged vs Overall', fontsize=11, fontweight='bold')

hv_geo = hv.groupby('Geography')['Exited'].mean()*100
axes[1].bar(hv_geo.index, hv_geo.values,
            color=['#3266ad','#c0392b','#f0a500'], edgecolor='white', width=0.5)
for i,(g,v) in enumerate(hv_geo.items()):
    axes[1].text(i, v+0.4, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('HV Disengaged by Geography', fontsize=11, fontweight='bold')

hv['AgeG'] = pd.cut(hv['Age'], bins=[0,25,35,45,55,120], labels=['18-25','26-35','36-45','46-55','56+'])
hv_age = hv.groupby('AgeG', observed=True)['Exited'].mean()*100
axes[2].bar(hv_age.index, hv_age.values, color='#c0392b', edgecolor='white', width=0.55)
for i,(a,v) in enumerate(hv_age.items()):
    axes[2].text(i, v+0.4, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Churn Rate (%)')
axes[2].set_title('HV Disengaged by Age Group', fontsize=11, fontweight='bold')
axes[2].tick_params(axis='x', labelsize=9)

plt.suptitle('High-Value Disengaged Customer Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'high_value_disengaged.png')
plt.show()

HIGH-VALUE DISENGAGED ANALYSIS
  Total HV disengaged    : 2,456
  Churn rate             : 32.3%
  Overall churn rate     : 20.4%
  Risk premium           : +12.0 pp
  Avg balance            : €130,926
  Total revenue @ risk   : €103.6M
  Germany share          : 41.9%
  Figure saved → e:\bank-retention-intelligence\reports/screenshots\high_value_disengaged.png


## 11. Retention Strategy Summary

In [12]:
print("\n" + "="*65)
print("  RETENTION STRATEGIES — DATA-BACKED RECOMMENDATIONS")
print("="*65)

strategies = [
    ("🔴 Strategy 1 — Immediate Outreach",
     f"{len(df[df['ChurnProbability']>0.80]):,} customers",
     "Churn prob >80%",
     "Assign relationship managers · Personal phone call · Premium offer"),
    ("🟠 Strategy 2 — Cross-Sell Programme",
     f"{len(df[df['NumOfProducts']==1]):,} customers",
     "1 product → 27.7% churn  |  2 products → 7.6% churn",
     "Product bundle offers · In-app cross-sell · Targeted emails"),
    ("🟡 Strategy 3 — Reactivation Campaign",
     f"{len(df[df['IsActiveMember']==0]):,} customers",
     "Inactive: 26.9% vs 14.3% for active",
     "Loyalty points · Push notifications · Personalised offers"),
    ("🟡 Strategy 4 — Germany Programme",
     f"{len(df[df['Geography']=='Germany']):,} customers",
     "Germany: 32.4% vs France: 16.2%",
     "Regional NPS survey · Local competitive analysis · Localised products"),
    ("🟡 Strategy 5 — Age 46–55 Advisory",
     f"{len(df[(df['Age']>=46)&(df['Age']<=55)]):,} customers",
     "50.6% churn rate — highest cohort",
     "Wealth planning · Premium advisory · Dedicated financial advisor"),
]

for title, target, insight, action in strategies:
    print(f"\n  {title}")
    print(f"    Target  : {target}")
    print(f"    Insight : {insight}")
    print(f"    Action  : {action}")

print("\n" + "="*65)


  RETENTION STRATEGIES — DATA-BACKED RECOMMENDATIONS

  🔴 Strategy 1 — Immediate Outreach
    Target  : 793 customers
    Insight : Churn prob >80%
    Action  : Assign relationship managers · Personal phone call · Premium offer

  🟠 Strategy 2 — Cross-Sell Programme
    Target  : 5,084 customers
    Insight : 1 product → 27.7% churn  |  2 products → 7.6% churn
    Action  : Product bundle offers · In-app cross-sell · Targeted emails

  🟡 Strategy 3 — Reactivation Campaign
    Target  : 4,849 customers
    Insight : Inactive: 26.9% vs 14.3% for active
    Action  : Loyalty points · Push notifications · Personalised offers

  🟡 Strategy 4 — Germany Programme
    Target  : 2,509 customers
    Insight : Germany: 32.4% vs France: 16.2%
    Action  : Regional NPS survey · Local competitive analysis · Localised products

  🟡 Strategy 5 — Age 46–55 Advisory
    Target  : 1,311 customers
    Insight : 50.6% churn rate — highest cohort
    Action  : Wealth planning · Premium advisory · Dedicat

## 12. Save Final Segmented Dataset & Insights

In [13]:
save_processed(df, 'final_segmented_dataset.csv')

segment_insights = seg_summary.copy()
segment_insights['AvgRSI'] = df.groupby('Cluster')['RSI'].mean().round(1).values
segment_insights['TopRecommendation'] = (
    df.groupby('Cluster')['Recommendation']
    .agg(lambda x: x.value_counts().index[0])
    .values
)
save_processed(segment_insights, 'segment_insights.csv')

print("\nFinal output files:")
print("  ✓ data/processed/final_segmented_dataset.csv")
print("  ✓ data/processed/segment_insights.csv")
print("  ✓ reports/screenshots/  (all charts)")
print("\nProject complete! Launch dashboard: streamlit run streamlit_app/app.py")

  Saved → e:\bank-retention-intelligence\data\processed\final_segmented_dataset.csv  (10,000 rows)
  Saved → e:\bank-retention-intelligence\data\processed\segment_insights.csv  (4 rows)

Final output files:
  ✓ data/processed/final_segmented_dataset.csv
  ✓ data/processed/segment_insights.csv
  ✓ reports/screenshots/  (all charts)

Project complete! Launch dashboard: streamlit run streamlit_app/app.py
